# Session 2: Introduction to Embeddings and Semantic Search

**Goal:** Understand how text becomes numbers (vectors) and use them to search by *meaning*.

| What you'll learn | Why it matters |
|---|---|
| Generate embeddings via API | Foundation of all semantic search |
| Cosine similarity | How machines measure "similar meaning" |
| When cosine **fails** | Critical pitfall from the lecture |
| Build a semantic search engine | Hands-on retrieval over real documents |

**Dataset:** 23 articles about the 2026 Iran War — a recent event **not in any model's training data**.

**Time:** ~25 minutes

## Setup

In [39]:
!pip install openai numpy scikit-learn -q

In [40]:
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict
import os

# ── Configure OpenRouter API ──
from google.colab import userdata
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

EMBEDDING_MODEL = "openai/text-embedding-3-small"  # 1536 dimensions

print(f"✓ Client ready — model: {EMBEDDING_MODEL}")

✓ Client ready — model: openai/text-embedding-3-small


## Load the Document Corpus

In [41]:
# ── Load the corpus ──
# Upload usa-iran-corpus.txt using the Files panel (📁) on the left,
# or run:  from google.colab import files; uploaded = files.upload()

def load_corpus(path="usa-iran-corpus.txt"):
    """Load documents and metadata from the corpus file."""
    with open(path) as f:
        raw = f.read()

    blocks = raw.split("===DOC===\n")[1:]  # skip text before first delimiter
    documents, metadatas = [], []

    for block in blocks:
        header_part, content = block.split("===\n", 1)
        meta = {}
        for line in header_part.strip().split("\n"):
            if ": " in line:
                key, val = line.split(": ", 1)
                meta[key.strip()] = val.strip()
        documents.append(content.strip())
        metadatas.append(meta)

    return documents, metadatas

documents, metadatas = load_corpus()
print(f"✓ Loaded {len(documents)} documents")
for i in range(3):
    title = documents[i].split("\n")[0][:70]
    print(f"  [{i}] ({metadatas[i].get('topic','?')}) {title}")
print(f"  ... and {len(documents) - 3} more")

✓ Loaded 23 documents
  [0] (overview) # The 2026 Iran War: Comprehensive Overview
  [1] (reporting) # US-Israel War on Iran: Day 25 of Attacks
  [2] (diplomatic) # Trump Says U.S. in Talks with Iran to End War; Iran Denies It
  ... and 20 more


## Part 1: What Are Embeddings?

An **embedding** converts text into a list of numbers (a *vector*) that captures its meaning.

```
"Iran launched missiles"  →  [0.12, -0.34, 0.56, …]   (1536 numbers)
"Tehran fired rockets"    →  [0.11, -0.33, 0.55, …]   (nearby in vector space)
"Oil prices rose sharply" →  [0.78, 0.21, -0.44, …]   (far away)
```

**Key idea:** Similar meaning → vectors point in similar directions → we can *search by meaning*.

In [42]:
def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> List[float]:
    """Generate embedding for a single text."""
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def get_embeddings_batch(texts: List[str], model: str = EMBEDDING_MODEL) -> List[List[float]]:
    """Generate embeddings for multiple texts in one API call."""
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]


print("✓ Embedding functions defined")

✓ Embedding functions defined


In [43]:
# Generate your first embedding
text = "The United States launched airstrikes against Iran"
embedding = get_embedding(text)

print(f"Text:       '{text}'")
print(f"Dimensions: {len(embedding)}")
print(f"First 8:    {[round(v, 4) for v in embedding[:8]]}")
print(f"\nThis vector captures the *meaning* of the sentence in {len(embedding)} numbers.")

Text:       'The United States launched airstrikes against Iran'
Dimensions: 1536
First 8:    [-0.0442, 0.0115, 0.0183, 0.0164, -0.0078, -0.0093, -0.0167, 0.0679]

This vector captures the *meaning* of the sentence in 1536 numbers.


## Part 2: Cosine Similarity

**Cosine similarity** measures the angle between two vectors:
- **1.0** = identical meaning
- **~0.8** = closely related
- **~0.5** = loosely related
- **~0.0** = unrelated

In [44]:
def calculate_similarity(emb1: List[float], emb2: List[float]) -> float:
    """Cosine similarity between two embeddings (0 to 1)."""
    a = np.array(emb1).reshape(1, -1)
    b = np.array(emb2).reshape(1, -1)
    return float(cosine_similarity(a, b)[0][0])


print("✓ Similarity function defined")

✓ Similarity function defined


In [45]:
# Compare texts about the Iran conflict
texts = [
    "The United States launched airstrikes against Iranian nuclear facilities",
    "American military forces bombed Iran's nuclear infrastructure",         # near-identical meaning
    "Oil prices surged after the Strait of Hormuz was blockaded",           # same conflict, different topic
    "Over 100 children died in a school strike in southern Iran",           # same conflict, humanitarian angle
    "Python is a popular programming language for data science",            # completely unrelated
]

embeddings = get_embeddings_batch(texts)
reference = embeddings[0]

print(f"Reference: '{texts[0]}'\n")
for i, (text, emb) in enumerate(zip(texts, embeddings)):
    score = calculate_similarity(reference, emb)
    bar = "█" * int(score * 40)
    print(f"  {score:.3f} {bar}")
    print(f"         '{text}'\n")

Reference: 'The United States launched airstrikes against Iranian nuclear facilities'

  1.000 ███████████████████████████████████████
         'The United States launched airstrikes against Iranian nuclear facilities'

  0.735 █████████████████████████████
         'American military forces bombed Iran's nuclear infrastructure'

  0.364 ██████████████
         'Oil prices surged after the Strait of Hormuz was blockaded'

  0.391 ███████████████
         'Over 100 children died in a school strike in southern Iran'

  0.039 █
         'Python is a popular programming language for data science'



### ⚠️ When Cosine Similarity Fails

From the lecture: cosine measures **structural similarity**, not **factual correctness**.

In [46]:
# These two sentences have OPPOSITE truth values but nearly identical embeddings
correct = "Tehran is the capital of Iran"
wrong   = "Isfahan is the capital of Iran"

emb_correct = get_embedding(correct)
emb_wrong   = get_embedding(wrong)

score = calculate_similarity(emb_correct, emb_wrong)
print(f"'{correct}'")
print(f"'{wrong}'")
print(f"\nSimilarity: {score:.3f}  ← Almost identical!")
print("\n→ Cosine is great for RETRIEVAL (finding related docs)")
print("→ But useless for checking if an answer is CORRECT")

'Tehran is the capital of Iran'
'Isfahan is the capital of Iran'

Similarity: 0.761  ← Almost identical!

→ Cosine is great for RETRIEVAL (finding related docs)
→ But useless for checking if an answer is CORRECT


## Part 3: Building a Semantic Search Engine

Now let's combine embeddings + cosine similarity into a search engine over the full corpus.

In [47]:
class SimpleSemanticSearch:
    """In-memory semantic search using cosine similarity."""

    def __init__(self):
        self.documents = []
        self.metadatas = []
        self.embeddings = []

    def add_documents(self, documents: List[str], metadatas: List[Dict] = None):
        print(f"Indexing {len(documents)} documents...")
        new_embeddings = get_embeddings_batch(documents)
        self.documents.extend(documents)
        self.embeddings.extend(new_embeddings)
        if metadatas:
            self.metadatas.extend(metadatas)
        else:
            self.metadatas.extend([{}] * len(documents))
        print(f"✓ Total indexed: {len(self.documents)} documents")

    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        query_emb = get_embedding(query)
        scored = []
        for i, doc_emb in enumerate(self.embeddings):
            score = calculate_similarity(query_emb, doc_emb)
            scored.append((i, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return [
            {"rank": r + 1, "document": self.documents[i], "score": s,
             "metadata": self.metadatas[i]}
            for r, (i, s) in enumerate(scored[:top_k])
        ]


print("✓ SimpleSemanticSearch class defined")

✓ SimpleSemanticSearch class defined


In [48]:
# Index the full corpus
engine = SimpleSemanticSearch()
engine.add_documents(documents, metadatas)

Indexing 23 documents...
✓ Total indexed: 23 documents


In [57]:
def show_results(query: str, results: List[Dict]):
    print(f"\nQuery: '{query}'")
    print("=" * 80)
    for r in results:
        source = r["metadata"].get("source", "?")
        topic = r["metadata"].get("topic", "?")
        # Show first line (title) + a snippet
        lines = r["document"].split("\n")
        title = lines[0][:70]
        print(f"\n  [{r['rank']}] Score: {r['score']:.3f}  ({topic}, {source})")
        print(f"      {title}")
        #text = "\n  ".join(lines[1:4])
        #print(f"  {text}")
    print()

In [58]:
# Search 1: Nuclear program
show_results(
    "What happened to Iran's nuclear program?",
    engine.search("What happened to Iran's nuclear program?")
)


Query: 'What happened to Iran's nuclear program?'

  [1] Score: 0.524  (military, Center for Strategic and International Studies (CSIS))
      # Operation Epic Fury and the Remnants of Iran's Nuclear Program

  [2] Score: 0.523  (nuclear, Arms Control Association)
      # The U.S. War on Iran: New and Lingering Nuclear Risks

  [3] Score: 0.496  (nuclear, Arms Control Association)
      # U.S. Negotiators Were Ill-Prepared for Serious Nuclear Negotiations 



In [52]:
# Search 2: Economic impact
show_results(
    "How did the war affect oil prices and the global economy?",
    engine.search("How did the war affect oil prices and the global economy?")
)


Query: 'How did the war affect oil prices and the global economy?'

  [1] Score: 0.546  (economic, Council on Foreign Relations (CFR))
      # The Iran War is Causing Energy Chaos in Asia

  [2] Score: 0.529  (economic, Federal Reserve Bank of Dallas)
      # Strait of Hormuz Closure: Global Economic Impact Analysis

  [3] Score: 0.523  (reporting, Al Jazeera)
      # Maritime Insurers Cancel War Risk Cover in Gulf: Impact on Energy Co



In [53]:
# Search 3: Civilian casualties
show_results(
    "Were there civilian casualties?",
    engine.search("Were there civilian casualties?")
)


Query: 'Were there civilian casualties?'

  [1] Score: 0.433  (humanitarian, Refugees International)
      # U.S./Israel-Iran War on Course for Cataclysmic Civilian Harm, Displa

  [2] Score: 0.409  (humanitarian, Amnesty International)
      # USA/Iran: US Strike on School That Killed Over 100 Children Must Fac

  [3] Score: 0.334  (diplomatic, The Soufan Center)
      # Lebanon Wracked by Spillover from the Iran War



In [54]:
# Search 4: Cyber warfare
show_results(
    "What cyber attacks occurred during the conflict?",
    engine.search("What cyber attacks occurred during the conflict?")
)


Query: 'What cyber attacks occurred during the conflict?'

  [1] Score: 0.595  (cyber, PBS NewsHour)
      # Iran-Linked Hackers Target U.S. and Allies Amid Ongoing Conflict

  [2] Score: 0.505  (cyber, Unit 42 / Palo Alto Networks)
      # Threat Brief: March 2026 Escalation of Cyber Risk Related to Iran

  [3] Score: 0.410  (military, Center for Strategic and International Studies (CSIS))
      # Operation Epic Fury and the Remnants of Iran's Nuclear Program



### 🔬 Your turn

Try your own queries below. Some ideas:
- "Who started the war?"
- "What role did Hezbollah play?"
- "Were there any negotiations?"
- "How was Lebanon affected?"

In [59]:
# ── Try your own query ──
your_query = "What role did Hezbollah play?"  # ← change this

results = engine.search(your_query, top_k=5)
show_results(your_query, results)


Query: 'What role did Hezbollah play?'

  [1] Score: 0.556  (diplomatic, The Christian Science Monitor)
      # Why Hezbollah Fighters Are Embracing an Unpopular and Costly War wit

  [2] Score: 0.529  (diplomatic, The Soufan Center)
      # Lebanon Wracked by Spillover from the Iran War

  [3] Score: 0.350  (overview, Britannica)
      # The 2026 Iran War: Comprehensive Overview

  [4] Score: 0.322  (military, Center for Strategic and International Studies (CSIS))
      # Operation Epic Fury and the Remnants of Iran's Nuclear Program

  [5] Score: 0.321  (reporting, Al Jazeera)
      # US-Israel War on Iran: Day 25 of Attacks



## Key Takeaways

| Concept | What you learned |
|---|---|
| **Embeddings** | Text → vector of numbers that captures meaning |
| **Cosine similarity** | Measures how similar two vectors are (0–1) |
| **Cosine failure** | High similarity ≠ factual correctness |
| **Semantic search** | Find documents by meaning, not keywords |

**Next up:** ChromaDB — a proper vector database with persistence, metadata filtering, and efficient search at scale.